In [1]:
# ============================================================
# METHOD 3: Extract with Progress Bar (Advanced)
# ============================================================
print("\n\n" + "="*60)
print("METHOD 3: Extract with Progress Bar (Advanced)")
print("="*60)
print("\n# Uncomment the code below for detailed extraction progress:\n")


# Uncomment for extraction with progress bar:
from tqdm import tqdm
import zipfile
import os # Need to import os if it's not imported elsewhere
from pathlib import Path # Need to import Path if it's not imported elsewhere

# Path to zip (UPDATE THIS to yolo_dataset_1024.zip)
ZIP_PATH = "/content/drive/MyDrive/PCB_Dataset/yolo_dataset_1024.zip"

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        members = zip_ref.namelist()
        print(f"📦 Extracting {len(members)} files...")
        for member in tqdm(members, desc="Extracting"):
            # Extract to /content/ which is standard for Colab
            zip_ref.extract(member, '/content/')
    print("✅ Done!")
else:
    print(f"❌ Zip not found: {ZIP_PATH}")




METHOD 3: Extract with Progress Bar (Advanced)

# Uncomment the code below for detailed extraction progress:

📦 Extracting 16633 files...


Extracting: 100%|██████████| 16633/16633 [00:47<00:00, 351.07it/s] 

✅ Done!


In [2]:
!pip install ultralytics

# ============================================================

# START TRAINING - PCB DEFECT DETECTION

# ============================================================

import torch

from ultralytics import YOLO

import os

# 1. Verify GPU

# ============================================================

print("="*60)

print("🔍 SYSTEM CHECK")

print("="*60)

print(f"✅ PyTorch: {torch.__version__}")

print(f"✅ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    device = 0

else:

    print("⚠️ No GPU! Using CPU (will be slow)")

    device = 'cpu'

# 2. Create data.yaml

# ============================================================

print("\n📝 Creating data.yaml...")

data_yaml_content = """# PCB Defect Detection Dataset

path: /content/yolo_dataset

train: train/images

val: val/images

test: test/images

nc: 6

names:

  0: Missing_hole

  1: Mouse_bite

  2: Open_circuit

  3: Short

  4: Spur

  5: Spurious_copper

"""


🔍 SYSTEM CHECK
✅ PyTorch: 2.9.0+cu126
✅ CUDA Available: False
⚠️ No GPU! Using CPU (will be slow)

📝 Creating data.yaml...


In [3]:
# ============================================================
# 1. Define and Write data.yaml to a file
# ============================================================

# Define the YAML content with correct structure and absolute paths
# NOTE: Removed 'scale' as it's non-standard for YOLOv8 data.yaml
data_yaml_content = """
names:
- missing_hole
- mouse_bite
- open_circuit
- short
- spur
- spurious_copper
nc: 6
train: /content/train/images
val: /content/val/images
test: /content/test/images
"""
# Define the path where the file will be saved
YAML_FILE_PATH = '/content/data.yaml'

# Write the content to the file
try:
    with open(YAML_FILE_PATH, 'w') as f:
        f.write(data_yaml_content)
    print(f"✅ data.yaml created at: {YAML_FILE_PATH}")
    print("\nContents:")
    print("------------------------------------------------------------")
    print(data_yaml_content.strip())
    print("--------------------------------------")
except FileNotFoundError:
    print(f"❌ Cannot create data.yaml. Ensure the directory {Path(YAML_FILE_PATH).parent} exists.")


✅ data.yaml created at: /content/data.yaml

Contents:
------------------------------------------------------------
names:
- missing_hole
- mouse_bite
- open_circuit
- short
- spur
- spurious_copper
nc: 6
train: /content/train/images
val: /content/val/images
test: /content/test/images
--------------------------------------


In [ ]:

# 3. Load Model

# ============================================================

print("\n📦 Loading YOLOv8l (Teacher Model)...")

model = YOLO('yolov8l.pt')

print("✅ Model loaded")

# 4. Configure Training

# ============================================================

print("\n" + "="*60)

print("🎯 TRAINING CONFIGURATION")

print("="*60)

print("   Epochs:        50")

print("   Image Size:    640")

print(f"   Batch Size:    16 (GPU)" if device == 0 else "   Batch Size:    8 (CPU)")

print("   Device:        " + ("GPU (T4/A100)" if device == 0 else "CPU"))

print("   Augmentation:  Mosaic + MixUp")

print("   Cache:         True (faster training)")

print("   Workers:       8")

print("="*60)

# 5. Start Training

# ============================================================

print("\n🚀 STARTING TRAINING...")

print("="*60 + "\n")

results = model.train(

    data='/content/data.yaml',

    epochs=50,

    imgsz=320,

    batch=16 if device == 0 else 8,

    device=device,

    # Augmentations

    mosaic=1.0,

    mixup=0.1,

    # Training settings

    name='teacher_pcb',

    patience=10,

    # Performance optimizations

    cache=True,        # Cache images in RAM

    workers=8,         # Data loading workers

    # Saving

    save=True,

    save_period=5,     # Save every 5 epochs

    # Visualization

    plots=True,

    verbose=True

)

print("\n" + "="*60)

print("✅ TRAINING COMPLETE!")

print("="*60)

# 6. Evaluate

# ============================================================

print("\n📊 EVALUATION")

print("="*60)

print("\n1️⃣ Validation Set:")

val_results = model.val(split='val')

print(f"   mAP50:     {val_results.box.map50:.4f}")

print(f"   mAP50-95:  {val_results.box.map:.4f}")

print(f"   Precision: {val_results.box.mp:.4f}")

print(f"   Recall:    {val_results.box.mr:.4f}")

print("\n2️⃣ Test Set:")

test_results = model.val(split='test')

print(f"   mAP50:     {test_results.box.map50:.4f}")

print(f"   mAP50-95:  {test_results.box.map:.4f}")

print(f"   Precision: {test_results.box.mp:.4f}")

print(f"   Recall:    {test_results.box.mr:.4f}")

# 7. Save to Google Drive

# ============================================================

print("\n💾 SAVING RESULTS TO GOOGLE DRIVE")

print("="*60)

# Create results folder in Drive

!mkdir -p "/content/drive/MyDrive/PCB_Training"

# Copy training results

!cp -r /content/runs/detect/teacher_pcb "/content/drive/MyDrive/PCB_Training/"

print("✅ Results saved to: /content/drive/MyDrive/PCB_Training/teacher_pcb/")

# 8. Export Model

# ============================================================

print("\n📦 EXPORTING MODEL")

print("="*60)

print("Exporting to TorchScript...")

model.export(format='torchscript')

print("✅ Model exported")

# 9. Display Results

# ============================================================

print("\n" + "="*60)

print("📊 FINAL SUMMARY")

print("="*60)

print(f"\n📂 Google Drive Location:")

print(f"   /content/drive/MyDrive/PCB_Training/teacher_pcb/")

print(f"\n📂 Important Files:")

print(f"   weights/best.pt          ← Use this for inference")

print(f"   weights/last.pt          ← Last checkpoint")

print(f"   results.png              ← Training curves")

print(f"   confusion_matrix.png     ← Model performance")

print(f"   val_batch0_pred.jpg      ← Prediction samples")

print("\n🎉 ALL DONE!")

print("="*60)




📦 Loading YOLOv8l (Teacher Model)...
✅ Model loaded

🎯 TRAINING CONFIGURATION
   Epochs:        50
   Image Size:    640
   Batch Size:    8 (CPU)
   Device:        CPU
   Augmentation:  Mosaic + MixUp
   Cache:         True (faster training)
   Workers:       8

🚀 STARTING TRAINING...

Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None,

In [ ]:

with open('/content/yolo_dataset/data.yaml', 'w') as f:

    f.write(data_yaml_content)

print("✅ data.yaml created")

# 3. Load Model

# ============================================================

print("\n📦 Loading YOLOv8l (Teacher Model)...")

model = YOLO('yolov8l.pt')

print("✅ Model loaded")

# 4. Configure Training

# ============================================================

print("\n" + "="*60)

print("🎯 TRAINING CONFIGURATION")

print("="*60)

print("   Epochs:        50")

print("   Image Size:    640")

print(f"   Batch Size:    16 (GPU)" if device == 0 else "   Batch Size:    8 (CPU)")

print("   Device:        " + ("GPU (T4/A100)" if device == 0 else "CPU"))

print("   Augmentation:  Mosaic + MixUp")

print("   Cache:         True (faster training)")

print("   Workers:       8")

print("="*60)

# 5. Start Training

# ============================================================

print("\n🚀 STARTING TRAINING...")

print("="*60 + "\n")

results = model.train(

    data='/content/yolo_dataset/data.yaml',

    epochs=50,

    imgsz=640,

    batch=16 if device == 0 else 8,

    device=device,

    # Augmentations

    mosaic=1.0,

    mixup=0.1,

    # Training settings

    name='teacher_pcb',

    patience=10,

    # Performance optimizations

    cache=True,        # Cache images in RAM

    workers=8,         # Data loading workers

    # Saving

    save=True,

    save_period=5,     # Save every 5 epochs

    # Visualization

    plots=True,

    verbose=True

)

print("\n" + "="*60)

print("✅ TRAINING COMPLETE!")

print("="*60)

# 6. Evaluate

# ============================================================

print("\n📊 EVALUATION")

print("="*60)

print("\n1️⃣ Validation Set:")

val_results = model.val(split='val')

print(f"   mAP50:     {val_results.box.map50:.4f}")

print(f"   mAP50-95:  {val_results.box.map:.4f}")

print(f"   Precision: {val_results.box.mp:.4f}")

print(f"   Recall:    {val_results.box.mr:.4f}")

print("\n2️⃣ Test Set:")

test_results = model.val(split='test')

print(f"   mAP50:     {test_results.box.map50:.4f}")

print(f"   mAP50-95:  {test_results.box.map:.4f}")

print(f"   Precision: {test_results.box.mp:.4f}")

print(f"   Recall:    {test_results.box.mr:.4f}")

# 7. Save to Google Drive

# ============================================================

print("\n💾 SAVING RESULTS TO GOOGLE DRIVE")

print("="*60)

# Create results folder in Drive

!mkdir -p "/content/drive/MyDrive/PCB_Training"

# Copy training results

!cp -r /content/runs/detect/teacher_pcb "/content/drive/MyDrive/PCB_Training/"

print("✅ Results saved to: /content/drive/MyDrive/PCB_Training/teacher_pcb/")

# 8. Export Model

# ============================================================

print("\n📦 EXPORTING MODEL")

print("="*60)

print("Exporting to TorchScript...")

model.export(format='torchscript')

print("✅ Model exported")

# 9. Display Results

# ============================================================

print("\n" + "="*60)

print("📊 FINAL SUMMARY")

print("="*60)

print(f"\n📂 Google Drive Location:")

print(f"   /content/drive/MyDrive/PCB_Training/teacher_pcb/")

print(f"\n📂 Important Files:")

print(f"   weights/best.pt          ← Use this for inference")

print(f"   weights/last.pt          ← Last checkpoint")

print(f"   results.png              ← Training curves")

print(f"   confusion_matrix.png     ← Model performance")

print(f"   val_batch0_pred.jpg      ← Prediction samples")

print("\n🎉 ALL DONE!")

print("="*60)



✅ data.yaml created

📦 Loading YOLOv8l (Teacher Model)...
✅ Model loaded

🎯 TRAINING CONFIGURATION
   Epochs:        50
   Image Size:    640
   Batch Size:    16 (GPU)
   Device:        GPU (T4/A100)
   Augmentation:  Mosaic + MixUp
   Cache:         True (faster training)
   Workers:       8

🚀 STARTING TRAINING...

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7,

In [ ]:
# Display training curves

from IPython.display import Image, display

import matplotlib.pyplot as plt

print("\n📈 TRAINING CURVES:")

results_img = '/content/runs/detect/teacher_pcb/results.png'

if os.path.exists(results_img):

    display(Image(filename=results_img))

else:

    print("⚠️ Results image not found")

print("\n🔍 CONFUSION MATRIX:")

confusion_img = '/content/runs/detect/teacher_pcb/confusion_matrix.png'

if os.path.exists(confusion_img):

    display(Image(filename=confusion_img))

else:

    print("⚠️ Confusion matrix not found")

print("\n✅ Check your Google Drive for all results!")

# distill_rkd_final.py

from ultralytics import YOLO

from ultralytics.models.yolo.detect import DetectionTrainer

from ultralytics.utils import DEFAULT_CFG

import torch

import torch.nn.functional as F

from torch.cuda.amp import autocast, GradScaler

# === 1. Custom Trainer with RKD ===

class RKDDistillationTrainer(DetectionTrainer):

    def get_loss(self, batch):

        # Forward

        preds = self.model(batch['img'])

        teacher_preds = self.teacher_model(batch['img'])[0]  # [B, 84, 8400]

        # YOLO Task Loss

        loss, loss_items = super().get_loss(batch)

        # RKD: Soft label distillation

        s_logits = preds[0][..., 4:]  # class logits

        t_logits = teacher_preds[..., 4:].detach()

        T = 4.0

        s_prob = F.softmax(s_logits / T, dim=-1)

        t_prob = F.softmax(t_logits / T, dim=-1)

        distill_loss = F.kl_div(

            s_prob.log(), t_prob, reduction='batchmean'

        ) * (T ** 2)

        total_loss = loss + 0.7 * distill_loss

        return total_loss, loss_items

# === 2. Load Models ===

teacher = YOLO('/content/runs/detect/teacher_pcb/weights/best.pt')

teacher.model.eval()

for p in teacher.model.parameters():

    p.requires_grad = False

student = YOLO('yolov8n.pt')

# === 3. Train with RKD ===

trainer = RKDDistillationTrainer(

    cfg=DEFAULT_CFG,

    overrides={

        'model': 'yolov8n.pt',

        'data': '/content/yolo_dataset/data.yaml',

        'epochs': 50,

        'batch': 32,

        'imgsz': 640,

        'device': 0,

        'project': 'runs/distill',

        'name': 'student_rkd',

        'exist_ok': True,

        'optimizer': 'AdamW',

        'lr0': 0.001,

        'close_mosaic': 10,

        'mixup': 0.1,

        'cache': True,

        'workers': 8,

    }

)

# Inject teacher

trainer.teacher_model = teacher.model

# Train

trainer.train()

print("RKD Distillation Complete!")

print("Model saved at: runs/distill/student_rkd/weights/best.pt")



📈 TRAINING CURVES:


NameError: name 'os' is not defined

In [ ]:

# eval_fixed.py

from ultralytics import YOLO

import time

import os

# Download a test image

!wget -q https://ultralytics.com/images/bus.jpg -O test.jpg

teacher = YOLO('/content/runs/detect/teacher_pcb/weights/best.pt')

student = YOLO('runs/distill/student_rkd/weights/best.pt')  # ← Your current student

# Test Set Results (already done)

t_res = teacher.val(split='test', plots=False)

s_res = student.val(split='test', plots=False)

# FPS on CPU

def fps(model, img='test.jpg', runs=100):

    model.to('cpu')

    start = time.time()

    for _ in range(runs):

        model.predict(img, verbose=False)

    return runs / (time.time() - start)

print(f"Teacher: mAP50-95 = {t_res.box.map:.3f}, FPS(CPU) = {fps(teacher):.1f}")

print(f"Student: mAP50-95 = {s_res.box.map:.3f}, FPS(CPU) = {fps(student):.1f}")

# eval_fixed.py

from ultralytics import YOLO

import time

import numpy as np

import torch

# Create dummy image

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

teacher = YOLO('/content/runs/detect/teacher_pcb/weights/best.pt')

student = YOLO('runs/distill/student_rkd/weights/best.pt')  # ← Current student

# Test mAP (already done)

print("Teacher mAP50-95 (test): 0.646")

print("Student mAP50-95 (test): 0.531")

# FPS on CPU

def fps_cpu(model, runs=100):

    model.model.to('cpu')

    model.model.eval()

    x = torch.from_numpy(dummy).permute(2, 0, 1).unsqueeze(0).float() / 255.0

    x = x.to('cpu')

    start = time.time()

    for _ in range(runs):

        with torch.no_grad():

            model.model(x)

    return runs / (time.time() - start)

print(f"Teacher FPS (CPU): {fps_cpu(teacher):.1f}")

print(f"Student FPS (CPU): {fps_cpu(student):.1f}")

import os

print("Student weights exist:")

print("best.pt:", os.path.exists('runs/distill/student_rkd_v2/weights/best.pt'))

print("last.pt:", os.path.exists('runs/distill/student_rkd_v2/weights/last.pt'))

In [ ]:
# ============================================================
# EXTRACT ZIP FILE IN GOOGLE COLAB
# ============================================================

import zipfile
import os
from pathlib import Path

# METHOD 1: Extract from Google Drive (Recommended if already uploaded)
# ============================================================
print("="*60)
print("METHOD 1: Extract from Google Drive")
print("="*60)

# Mount Google Drive
print("\n📁 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

# Path to your zip file in Google Drive
# UPDATE THIS PATH to match where you uploaded your zip
ZIP_PATH = "/content/drive/MyDrive/PCB_Dataset/yolo_dataset.zip"

# Check if zip exists
if os.path.exists(ZIP_PATH):
    print(f"\n✅ Found zip file: {ZIP_PATH}")

    # Extract to Colab local storage
    print("\n📦 Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')

    print("✅ Extraction complete!")

    # Verify extraction
    print("\n🔍 Verifying extracted files...")
    extracted_path = Path('/content/yolo_dataset')

    if extracted_path.exists():
        print(f"✅ Dataset extracted to: {extracted_path}")

        # List contents
        print("\n📂 Dataset structure:")
        for item in sorted(extracted_path.iterdir()):
            if item.is_dir():
                file_count = len(list(item.rglob('*')))
                print(f"   📁 {item.name}/ ({file_count} items)")
            else:
                size = item.stat().st_size / 1024  # KB
                print(f"   📄 {item.name} ({size:.2f} KB)")

        # Check for required folders
        print("\n✅ Checking dataset structure...")
        required_folders = ['train/images', 'train/labels',
                          'val/images', 'val/labels',
                          'test/images', 'test/labels']

        all_exist = True
        for folder in required_folders:
            folder_path = extracted_path / folder
            if folder_path.exists():
                count = len(list(folder_path.glob('*')))
                print(f"   ✅ {folder:20s} → {count} files")
            else:
                print(f"   ❌ {folder:20s} → MISSING!")
                all_exist = False

        if all_exist:
            print("\n🎉 Dataset is ready for training!")
        else:
            print("\n⚠️ Some folders are missing. Check your zip structure.")
    else:
        print("❌ Extraction failed. Check your zip file structure.")

else:
    print(f"❌ Zip file not found at: {ZIP_PATH}")
    print("\n📋 Available files in MyDrive/PCB_Dataset:")
    pcb_folder = Path("/content/drive/MyDrive/PCB_Dataset")
    if pcb_folder.exists():
        for item in pcb_folder.iterdir():
            print(f"   - {item.name}")
    else:
        print("   ❌ Folder doesn't exist. Please create it and upload your zip.")

    print("\n" + "="*60)
    print("OR USE METHOD 2 BELOW (Direct Upload)")
    print("="*60)


# ============================================================
# METHOD 2: Direct Upload to Colab (Alternative)
# Use this if you haven't uploaded to Drive yet
# ============================================================
print("\n\n" + "="*60)
print("METHOD 2: Direct Upload from Your PC")
print("="*60)
print("\n⚠️ Only run this if METHOD 1 didn't work")
print("⚠️ This will ask you to upload yolo_dataset.zip from your PC")
print("\n# Uncomment the code below to use this method:\n")

"""
# Uncomment this section to use direct upload:

from google.colab import files
import zipfile

print("📤 Please select yolo_dataset.zip from your PC...")
uploaded = files.upload()

# Get the uploaded filename
zip_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {zip_filename}")

# Extract
print(f"\n📦 Extracting {zip_filename}...")
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    # Show extraction progress
    members = zip_ref.namelist()
    print(f"   Total files to extract: {len(members)}")

    for i, member in enumerate(members):
        zip_ref.extract(member, '/content/')
        if (i + 1) % 100 == 0:
            print(f"   Extracted {i + 1}/{len(members)} files...")

print("✅ Extraction complete!")

# Verify
extracted_path = Path('/content/yolo_dataset')
if extracted_path.exists():
    print(f"✅ Dataset ready at: {extracted_path}")
else:
    print("⚠️ Dataset folder not found. Check zip structure.")
"""


# ============================================================
# METHOD 3: Extract with Progress Bar (Advanced)
# ============================================================
print("\n\n" + "="*60)
print("METHOD 3: Extract with Progress Bar (Advanced)")
print("="*60)
print("\n# Uncomment the code below for detailed extraction progress:\n")

"""
# Uncomment for extraction with progress bar:

from tqdm import tqdm
import zipfile

# Path to zip (update this)
ZIP_PATH = "/content/drive/MyDrive/PCB_Dataset/yolo_dataset.zip"

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        members = zip_ref.namelist()

        print(f"📦 Extracting {len(members)} files...")
        for member in tqdm(members, desc="Extracting"):
            zip_ref.extract(member, '/content/')

    print("✅ Done!")
else:
    print(f"❌ Zip not found: {ZIP_PATH}")
"""


# ============================================================
# HELPER: List Drive Contents
# ============================================================
print("\n\n" + "="*60)
print("HELPER: Show Google Drive Contents")
print("="*60)

def show_drive_contents(path="/content/drive/MyDrive"):
    """Show all folders and files in Google Drive"""
    print(f"\n📂 Contents of: {path}")
    try:
        p = Path(path)
        if p.exists():
            items = sorted(p.iterdir())
            for item in items[:20]:  # Show first 20 items
                if item.is_dir():
                    print(f"   📁 {item.name}/")
                else:
                    size = item.stat().st_size / (1024*1024)  # MB
                    print(f"   📄 {item.name} ({size:.2f} MB)")

            if len(items) > 20:
                print(f"   ... and {len(items) - 20} more items")
        else:
            print(f"   ❌ Path doesn't exist")
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Show MyDrive root
show_drive_contents("/content/drive/MyDrive")

# Show PCB_Dataset folder (if it exists)
show_drive_contents("/content/drive/MyDrive/PCB_Dataset")


# ============================================================
# FINAL CHECK
# ============================================================
print("\n\n" + "="*60)
print("FINAL CHECK")
print("="*60)

dataset_path = Path('/content/yolo_dataset')
if dataset_path.exists():
    print("✅ Dataset is ready at: /content/yolo_dataset")

    # Count files
    total_images = 0
    total_labels = 0

    for split in ['train', 'val', 'test']:
        img_path = dataset_path / split / 'images'
        lbl_path = dataset_path / split / 'labels'

        if img_path.exists():
            imgs = len(list(img_path.glob('*.jpg'))) + len(list(img_path.glob('*.png')))
            total_images += imgs

        if lbl_path.exists():
            lbls = len(list(lbl_path.glob('*.txt')))
            total_labels += lbls

    print(f"   📊 Total: {total_images} images, {total_labels} labels")
    print("\n🎉 Ready to proceed with training!")

else:
    print("❌ Dataset not extracted yet")
    print("\n💡 Next steps:")
    print("   1. Make sure yolo_dataset.zip is in Google Drive")
    print("   2. Update ZIP_PATH in METHOD 1 above")
    print("   3. Run this cell again")
    print("   OR")
    print("   4. Use METHOD 2 (direct upload)")

METHOD 1: Extract from Google Drive

📁 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted

✅ Found zip file: /content/drive/MyDrive/PCB_Dataset/yolo_dataset.zip

📦 Extracting dataset...
✅ Extraction complete!

🔍 Verifying extracted files...
✅ Dataset extracted to: /content/yolo_dataset

📂 Dataset structure:
   📄 data.yaml (0.79 KB)
   📁 test/ (558 items)
   📁 train/ (4437 items)
   📁 val/ (557 items)

✅ Checking dataset structure...
   ✅ train/images         → 2217 files
   ✅ train/labels         → 2217 files
   ✅ val/images           → 277 files
   ✅ val/labels           → 277 files
   ✅ test/images          → 278 files
   ✅ test/labels          → 278 files

🎉 Dataset is ready for training!


METHOD 2: Direct Upload from Your PC

⚠️ Only run this if METHOD 1 didn't work
⚠️ This will ask you to upload yolo_dataset.zip from your PC

# Uncomment the code below to 

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.6 MB/s eta 0:00:00


In [ ]:
# ============================================================
# START TRAINING - PCB DEFECT DETECTION
# ============================================================

import torch
from ultralytics import YOLO
import os

# 1. Verify GPU
# ============================================================
print("="*60)
print("🔍 SYSTEM CHECK")
print("="*60)
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = 0
else:
    print("⚠️ No GPU! Using CPU (will be slow)")
    device = 'cpu'

# 2. Create data.yaml
# ============================================================
print("\n📝 Creating data.yaml...")
data_yaml_content = """# PCB Defect Detection Dataset
path: /content/yolo_dataset
train: train/images
val: val/images
test: test/images

nc: 6

names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copper
"""

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content)
print("✅ data.yaml created")

# 3. Load Model
# ============================================================
print("\n📦 Loading YOLOv8l (Teacher Model)...")
model = YOLO('yolov8l.pt')
print("✅ Model loaded")

# 4. Configure Training
# ============================================================
print("\n" + "="*60)
print("🎯 TRAINING CONFIGURATION")
print("="*60)
print("   Epochs:        50")
print("   Image Size:    640")
print(f"   Batch Size:    16 (GPU)" if device == 0 else "   Batch Size:    8 (CPU)")
print("   Device:        " + ("GPU (T4/A100)" if device == 0 else "CPU"))
print("   Augmentation:  Mosaic + MixUp")
print("   Cache:         True (faster training)")
print("   Workers:       8")
print("="*60)

# 5. Start Training
# ============================================================
print("\n🚀 STARTING TRAINING...")
print("="*60 + "\n")

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16 if device == 0 else 8,
    device=device,

    # Augmentations
    mosaic=1.0,
    mixup=0.1,

    # Training settings
    name='teacher_pcb',
    patience=10,

    # Performance optimizations
    cache=True,        # Cache images in RAM
    workers=8,         # Data loading workers

    # Saving
    save=True,
    save_period=5,     # Save every 5 epochs

    # Visualization
    plots=True,
    verbose=True
)

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)

# 6. Evaluate
# ============================================================
print("\n📊 EVALUATION")
print("="*60)

print("\n1️⃣ Validation Set:")
val_results = model.val(split='val')
print(f"   mAP50:     {val_results.box.map50:.4f}")
print(f"   mAP50-95:  {val_results.box.map:.4f}")
print(f"   Precision: {val_results.box.mp:.4f}")
print(f"   Recall:    {val_results.box.mr:.4f}")

print("\n2️⃣ Test Set:")
test_results = model.val(split='test')
print(f"   mAP50:     {test_results.box.map50:.4f}")
print(f"   mAP50-95:  {test_results.box.map:.4f}")
print(f"   Precision: {test_results.box.mp:.4f}")
print(f"   Recall:    {test_results.box.mr:.4f}")

# 7. Save to Google Drive
# ============================================================
print("\n💾 SAVING RESULTS TO GOOGLE DRIVE")
print("="*60)

# Create results folder in Drive
!mkdir -p "/content/drive/MyDrive/PCB_Training_Results"

# Copy training results
!cp -r /content/runs/detect/teacher_pcb "/content/drive/MyDrive/PCB_Training_Results/"

print("✅ Results saved to: /content/drive/MyDrive/PCB_Training_Results/teacher_pcb/")

# 8. Export Model
# ============================================================
print("\n📦 EXPORTING MODEL")
print("="*60)

print("Exporting to TorchScript...")
model.export(format='torchscript')
print("✅ Model exported")

# 9. Display Results
# ============================================================
print("\n" + "="*60)
print("📊 FINAL SUMMARY")
print("="*60)
print(f"\n📂 Google Drive Location:")
print(f"   /content/drive/MyDrive/PCB_Training_Results/teacher_pcb/")
print(f"\n📂 Important Files:")
print(f"   weights/best.pt          ← Use this for inference")
print(f"   weights/last.pt          ← Last checkpoint")
print(f"   results.png              ← Training curves")
print(f"   confusion_matrix.png     ← Model performance")
print(f"   val_batch0_pred.jpg      ← Prediction samples")
print("\n🎉 ALL DONE!")
print("="*60)

# Display training curves
from IPython.display import Image, display
import matplotlib.pyplot as plt

print("\n📈 TRAINING CURVES:")
results_img = '/content/runs/detect/teacher_pcb/results.png'
if os.path.exists(results_img):
    display(Image(filename=results_img))
else:
    print("⚠️ Results image not found")

print("\n🔍 CONFUSION MATRIX:")
confusion_img = '/content/runs/detect/teacher_pcb/confusion_matrix.png'
if os.path.exists(confusion_img):
    display(Image(filename=confusion_img))
else:
    print("⚠️ Confusion matrix not found")

print("\n✅ Check your Google Drive for all results!")

🔍 SYSTEM CHECK
✅ PyTorch: 2.8.0+cu126
✅ CUDA Available: False
⚠️ No GPU! Using CPU (will be slow)

📝 Creating data.yaml...


FileNotFoundError: [Errno 2] No such file or directory: '/content/yolo_dataset/data.yaml'

In [ ]:
# Check current YAML
!cat /content/yolo_dataset/data.yaml

# Overwrite it with correct paths
yaml_content = """path: /content/yolo_dataset
train: /content/yolo_dataset/train/images
val: /content/yolo_dataset/val/images
test: /content/yolo_dataset/test/images
nc: 6
names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copper

"""

with open("/content/yolo_dataset/data.yaml", "w") as f:
    f.write(yaml_content)

# Verify
!cat /content/yolo_dataset/data.yaml


# PCB Defect Detection Dataset Configuration
# Save this file as: E:\IBA_MS_DS 2026\Computer Vision\Project\PCB_defect_components\PCB_DATASET\yolo_dataset\data.yaml

# IMPORTANT: Using absolute paths to avoid path resolution issues
path: E:\IBA_MS_DS 2026\Computer Vision\Project\PCB_defect_components\PCB_DATASET\yolo_dataset
train: E:\IBA_MS_DS 2026\Computer Vision\Project\PCB_defect_components\PCB_DATASET\yolo_dataset\train\images
val: E:\IBA_MS_DS 2026\Computer Vision\Project\PCB_defect_components\PCB_DATASET\yolo_dataset\val\images
test: E:\IBA_MS_DS 2026\Computer Vision\Project\PCB_defect_components\PCB_DATASET\yolo_dataset\test\images

# Number of classes
nc: 6

# Class names
names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copperpath: /content/yolo_dataset
train: /content/yolo_dataset/train/images
val: /content/yolo_dataset/val/images
test: /content/yolo_dataset/test/images
nc: 6
names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circu

In [ ]:
# distill_rkd_final.py
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import DEFAULT_CFG
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
import torch
from ultralytics import YOLO
import os

# === 1. Custom Trainer with RKD ===
class RKDDistillationTrainer(DetectionTrainer):
    def get_loss(self, batch):
        # Forward
        preds = self.model(batch['img'])
        teacher_preds = self.teacher_model(batch['img'])[0]  # [B, 84, 8400]

        # YOLO Task Loss
        loss, loss_items = super().get_loss(batch)

        # RKD: Soft label distillation
        s_logits = preds[0][..., 4:]  # class logits
        t_logits = teacher_preds[..., 4:].detach()

        T = 4.0
        s_prob = F.softmax(s_logits / T, dim=-1)
        t_prob = F.softmax(t_logits / T, dim=-1)

        distill_loss = F.kl_div(
            s_prob.log(), t_prob, reduction='batchmean'
        ) * (T ** 2)

        total_loss = loss + 0.7 * distill_loss
        return total_loss, loss_items

# === 2. Load Models ===
teacher = YOLO('/content/drive/MyDrive/PCB_Training_Results/teacher_pcb/weights/best.pt')
teacher.model.eval()
for p in teacher.model.parameters():
    p.requires_grad = False

student = YOLO('yolov8n.pt')

# === 3. Train with RKD ===
trainer = RKDDistillationTrainer(
    cfg=DEFAULT_CFG,
    overrides={
        'model': 'yolov8n.pt',
        'data': '/content/yolo_dataset/data.yaml',
        'epochs': 50,
        'batch': 32,
        'imgsz': 640,
        'device': 'cpu',
        'project': 'runs/distill',
        'name': 'student_rkd',
        'exist_ok': True,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'close_mosaic': 10,
        'mixup': 0.1,
        'cache': True,
        'workers': 8,
    }
)

# Inject teacher
trainer.teacher_model = teacher.model

# Train
trainer.train()

print("RKD Distillation Complete!")
print("Model saved at: runs/distill/student_rkd/weights/best.pt")

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=student_rkd, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, perspective=0.0, plots=T

KeyboardInterrupt: 

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
teacher_path = '/content/drive/MyDrive/PCB_Training_Results/teacher_pcb/weights/best.pt'  # Update with exact Drive path
teacher = YOLO(teacher_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from ultralytics import YOLO
import time
import numpy as np
import torch
import cv2  # Added for image loading
from google.colab import files  # Added for file upload

# Upload image from system
print("Upload an image from your system for FPS benchmarking:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]  # Get the filename of the uploaded image

# Load models
teacher = YOLO(teacher_path)
student = YOLO('runs/distill/student_rkd/weights/best.pt')  # ← Current student


# FPS on CPU (modified to use uploaded image)
def fps_cpu(model, image_path, runs=100):
    # Load and preprocess the uploaded image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Failed to load image. Ensure it's a valid format (e.g., JPG, PNG).")
    img = cv2.resize(img, (640, 640))  # Resize to model input size
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    x = x.to('cpu')

    model.model.to('cpu')
    model.model.eval()

    start = time.time()
    for _ in range(runs):
        with torch.no_grad():
            model.model(x)
    return runs / (time.time() - start)

print(f"Teacher FPS (CPU): {fps_cpu(teacher, image_path):.1f}")
print(f"Student FPS (CPU): {fps_cpu(student, image_path):.1f}")

import os
print("Student weights exist:")
print("best.pt:", os.path.exists('runs/distill/student_rkd_v2/weights/best.pt'))
print("last.pt:", os.path.exists('runs/distill/student_rkd_v2/weights/last.pt'))

Upload an image from your system for FPS benchmarking:


Saving 01_missing_hole_01_geom.jpg to 01_missing_hole_01_geom (1).jpg


FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/teacher_pcb/weights/best.pt'

In [ ]:
# eval_fixed.py
from ultralytics import YOLO
import time
import os

# Download a test image
!wget -q https://ultralytics.com/images/bus.jpg -O test.jpg

teacher = YOLO('/content/runs/detect/teacher_pcb/weights/best.pt')
student = YOLO('runs/distill/student_rkd/weights/best.pt')  # ← Your current student

# Test Set Results (already done)
t_res = teacher.val(split='test', plots=False)
s_res = student.val(split='test', plots=False)

# FPS on CPU
def fps(model, img='test.jpg', runs=100):
    model.to('cpu')
    start = time.time()
    for _ in range(runs):
        model.predict(img, verbose=False)
    return runs / (time.time() - start)

print(f"Teacher: mAP50-95 = {t_res.box.map:.3f}, FPS(CPU) = {fps(teacher):.1f}")
print(f"Student: mAP50-95 = {s_res.box.map:.3f}, FPS(CPU) = {fps(student):.1f}")

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 112 layers, 43,611,234 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3247.7±562.3 MB/s, size: 1421.8 KB)
val: Scanning /content/yolo_dataset/test/labels.cache... 278 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 278/278 480.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.2it/s 14.7s
                   all        278       1178      0.969      0.917      0.959      0.646
          Missing_hole         46        203      0.969      0.985      0.989      0.733
            Mouse_bite         46        201      0.978      0.898       0.96      0.624
          Open_circuit         47        190      0.968      0.926      0.968       0.63
                 Short         46        189      0.957      0.899      0.941      0.666
                  Spur         46    

In [ ]:
# eval_fixed.py
from ultralytics import YOLO
import time
import numpy as np
import torch

# Create dummy image
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

teacher = YOLO('/content/runs/detect/teacher_pcb/weights/best.pt')
student = YOLO('runs/distill/student_rkd/weights/best.pt')  # ← Current student

# Test mAP (already done)
print("Teacher mAP50-95 (test): 0.646")
print("Student mAP50-95 (test): 0.531")

# FPS on CPU
def fps_cpu(model, runs=100):
    model.model.to('cpu')
    model.model.eval()
    x = torch.from_numpy(dummy).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    x = x.to('cpu')
    start = time.time()
    for _ in range(runs):
        with torch.no_grad():
            model.model(x)
    return runs / (time.time() - start)

print(f"Teacher FPS (CPU): {fps_cpu(teacher):.1f}")
print(f"Student FPS (CPU): {fps_cpu(student):.1f}")

Teacher mAP50-95 (test): 0.646
Student mAP50-95 (test): 0.531


KeyboardInterrupt: 

In [ ]:
import os
print("Student weights exist:")
print("best.pt:", os.path.exists('runs/distill/student_rkd_v2/weights/best.pt'))
print("last.pt:", os.path.exists('runs/distill/student_rkd_v2/weights/last.pt'))

Student weights exist:
best.pt: False
last.pt: False
